# Primera lectura CSV y limpieza de los mismos

La lectura y primera limpieza de los CSV se harán por grupos, atendiendo cada grupo de CSV de una manera separada pero con mismas directrices básicas. En el caso de tener que perfilar cada tabla de manera independiente se comentará detalladamente.

Los subgrupos son:
1. Tablas soporte préstamos
2. Tablas soporte catálogo
3. Población y premios
4. Directorio bibliotecas
5. Catálogo: registros y ejemplares


Posteriormente a la primera limpieza se realizará la limpieza independiente de cada tabla generada. Una vez terminada, se guardan DFs en formato parquet. Se guardarán en carpeta silver.

En el tratamiento independiente de cada tabla se observará:
1. Duplicados completos
2. Duplicados parciales
3. Comprobación valores nulos junto con asignación de valor real o nulo explícito
4. Asignación de los tipos de datos correctos.
    - Si pocos valores únicos repetidos en diversas filas -> category
    - Datos texto -> formato string pyarrow, para reducir memoria y velocidad en operaciones str.
    - Fechas -> Datetime
    - Solo años -> Int64
5. Normalización máxima de los campos de título, autor e ISBN en tablas registros y préstamos, eliminando signos y símbolos de puntuación
6. Eliminación de las columnas no necesarias


# Acciones iniciales

In [ ]:
from google.colab import drive # montar drive
import shutil # copiar docs. drive en disco local
from pathlib import Path# Gestionar rutas archivos
import json # Para crear y tratar archivo json
import pandas as pd
#modulo estandar de python. Counter (calse) cuenta cuantas veces aparece un
# elemento en un iterable:
from collections import Counter
from collections.abc import Iterable # Para poder iterar las rutas
import re # modulo de expresiones regulares
import unicodedata # Para normalizar strings
from IPython.display import display

pd.options.mode.copy_on_write = True
pd.set_option("io.parquet.engine", "pyarrow")
pd.set_option("mode.string_storage", "pyarrow")

In [ ]:
# Montar drive:
drive.mount('/content/drive')

## Constantes


In [ ]:
# 1. Rutas originales
pf = Path('drive')
local = Path('/content')

# 2. Rutas sobre las que trabajar:
# 2.1 Rutas que apuntan hacia Drive
DRIVE_ORIGEN = pf / 'bronze'
DRIVE_DESTINO = pf / 'silver'

# 2.2 Rutas que apuntan hacia disco local:
ORIGEN_LOCAL = local / 'datos_bronze'
DESTINO_LOCAL = local / 'resultados_silver'
ORIGEN_LOCAL.mkdir(parents=True, exist_ok=True)
DESTINO_LOCAL.mkdir(parents=True, exist_ok=True)


# 3. Lectura de los CSV, parametros extra a tener en cuenta al leerlos.
PARAMS_BRONZE = {'dtype': 'string', 'keep_default_na': False}

# 4. Limpieza y transformaciones DF
PATRONES_NULOS = ("", "-", ".", "?", "n/a", "na", "n.a", "n.a.",
                  "null", "none", "nan","sd", "s/d", "sin datos",
                  "tbd", "x", "xx", "xxx")

## Copia archivos drive en local

In [ ]:
# Copiar todas las subcarpetas de origen drive
#pudiendo excluir algunas.

EXCLUIR = ["PRESTAMOS_HISTORICOS"]
assert DRIVE_ORIGEN.exists(), f"No existe {DRIVE_ORIGEN}"

disponibles = sorted(p.name for p in DRIVE_ORIGEN.iterdir() if p.is_dir())
seleccion = [s for s in disponibles if s not in EXCLUIR]

faltan = [s for s in seleccion if s not in disponibles]
assert not faltan, f"No existen en DRIVE_ORIGEN: {faltan}. Disponibles: {disponibles}"

for sub in seleccion:
    shutil.copytree(DRIVE_ORIGEN / sub, ORIGEN_LOCAL / sub, dirs_exist_ok=True)

DESTINO_LOCAL.mkdir(parents=True, exist_ok=True)

print(f"Subcarpetas en Drive: {disponibles}")
print(f"Copiadas: {seleccion}\n")

for sub in seleccion:
    csvs = sorted(p.name for p in (ORIGEN_LOCAL / sub).rglob("*.csv"))
    print(f"{sub} ({len(csvs)} csv): {csvs}")

In [ ]:
# Importacion archivo JSON desde Drive:

jsons = list(DRIVE_ORIGEN.glob('*.json'))
assert jsons, f'No hay ningún .json en {DRIVE_ORIGEN}'
for f in jsons:
    shutil.copy2(f, ORIGEN_LOCAL / f.name)
print(f"JSON copiados: {[f.name for f in jsons]}")


# Lectura archivo JSON para emplarlo en lectura CSV:

with open(ORIGEN_LOCAL / 'parametros_csv_brutos.json', encoding='utf-8') as f:
    params_json = json.load(f)

# Funciones

## Función resumen carpetas importadas de Drive y CSV incluidos.



In [ ]:
# COMPROBAR CANTIDAD CSVs EN CARPETAS, SUBCARPETAS Y REGISTROS EN ARCHIVO JSON
def mostrar_resumen(df: pd.DataFrame, carpetas: list[str]) -> None:
    """
    Imprime la información obtenida en la función resumen_carpetas_inicio.
    """
    print(f"Se han importado las siguientes carpetas: \n{carpetas}\n")
    print("\nTabla resumen de CSVs encontrados en cada carpeta y coincidencia "
    "en las claves del diccionario JSON, así como sus diferencias:\n")
    display(df)


def resumen_carpetas_inicio(ruta: Path,
                            params: dict) -> list[str]:
    """
    Resume la cantidad de CSV que hay por carpeta y la coincidencia de las
    extensiones en el archivo JSON, mostrando diferencias de haberlas.
    Retorna lista de nombres carpetas presentes y muestra el resumen de CSVs
    """
    csvs = [
        f for f in ruta.rglob('*') if f.is_file() and f.suffix.lower() == '.csv'
        ]
    en_drive = Counter(f.parent.relative_to(ruta).as_posix() for f in csvs)

    # JSON: carpeta de cada clave registrada
    en_json = Counter(Path(r).parent.as_posix() for r in params)

    cantidad_csv = (pd.DataFrame({'en_drive': en_drive, 'en_json': en_json})
                  .fillna(0).astype(int)
                  .assign(diferencia=lambda d: d['en_drive'] - d['en_json'])
                  .reset_index(names=["Sub_carpeta"]))
    carpetas = cantidad_csv.loc[cantidad_csv["en_drive"] !=0, 'Sub_carpeta' ].tolist()
    mostrar_resumen(cantidad_csv, carpetas)
    return carpetas

## Lectura CSV

Cada CSV se lee según los parámetros encontrados en la detección del encoding, guardados en el archivo JSON y parámetros extras de la lectura bronze (PARAMS_BRONZE), tipo dato de las columnas string y sin establecer valores nulos.

In [ ]:
def comprobar_nombre_carpetas(carpeta: str | list[str] |
                              tuple[str, ...] | set[str],
                              carpetas_disponibles: list[str]) -> list[str]:
    """
    Comprueba que la carpeta o lista de carpetas de las que extraer csv
    se encuentre en la lista de carpetas a utilizar en el análisis.
    Si se pasa lista de nombre, con el nombre repetido, se elimina duplicado.
    Lanza error si:
    - tipo de variable no str o list[str], tuple[str],set[str]
    - nombre carpetas no en lista de las disponibles.
    """
    # Comprobacion tipo variable
    if isinstance(carpeta, str):
      carpetas = [carpeta]
    elif (isinstance(carpeta, (list, tuple, set))
          and all(isinstance(c, str) for c in carpeta)):
      carpetas = list(dict.fromkeys(carpeta)) # quita duplicados
      if not carpetas:
        raise ValueError("No se ha indicado ninguna carpeta.")
    else:
      raise TypeError(
          f"'carpeta' debe ser str, list[str], set o tuple; "
          f"se recibió {type(carpeta).__name__}"
          )
    # Comprobacion nombre carpetas:
    nombres_erroneos = [c for c in carpetas if c not in carpetas_disponibles]
    if nombres_erroneos:
      raise ValueError(
          f"Carpetas con nombres no coincidentes: {nombres_erroneos}."
          f"\n Opciones válidas: {carpetas_disponibles}"
          )
    return carpetas


def leer_csv(
    params: dict[str, dict],
    rutas: Iterable[Path],
    raiz: Path) -> tuple[dict[str, pd.DataFrame], dict[str, str]]:
    """
    Función que realiza la lectura y conversion a DF de cada csv
    presente en las rutas seleccionadas.
    Retorna:
    - Diccionario que contiene el  nombre_archivo : DF.
    Se toma el nombre del archivo y no la ruta relativa,
    para facilitar el trato de los distintos DFs.
    - Diccionario con registro de errores nombre_ruta_relativa : error.
    En este caso, se fija el nombre de la ruta relativa
    para poder señalar ubicaión exacta del archivo erroneo.
    """
    d_dfs, d_fallos = {}, {} # diccionarios para almacenar info final

    for ruta in rutas:
      final_ruta = ruta.relative_to(raiz).as_posix()
      nombre_archivo = ruta.stem
      info_params = params.get(final_ruta)

      if info_params is None:
        d_fallos[final_ruta] = 'sin parámetros en el JSON'
        continue
      if info_params.get('error'):
        d_fallos[final_ruta] = info_params['error']
        continue
      if nombre_archivo in d_dfs:
        d_fallos[final_ruta] = f"Archivo {nombre_archivo}, ya leeido en otra carpeta"
        continue
      try:
        d_dfs[nombre_archivo] = pd.read_csv(
            ruta, **info_params['read_kwargs'], **PARAMS_BRONZE
            )
      except Exception as e:
        d_fallos[final_ruta] = repr(e)

    return d_dfs, d_fallos


def resumen_valores_lectura(dict_dfs: dict[str, pd.DataFrame],
                            dict_fallos: dict[str, str]) -> None:
    """
    Resume los csv leidos y convertidos en df.
    Imprimiendo nombre csv y shape del df resultante.
    Asi como los fallos posibles ocurridos.
    """
    print(f'CSV correctamente leidos ({len(dict_dfs)})\n '
          f'Nombre y Dimensiones de los DF: ')
    for k, df in dict_dfs.items():
      print(f'{k}: {df.shape[0]} filas y {df.shape[1]} columnas')
    if dict_fallos:
      print('CSV NO CONVERTIDOS:')
      for k, motivo in dict_fallos.items():
        print(f'{k}: {motivo}')


def lectura_bronze(params: dict[str, dict],
                   carpeta: str | list[str] |
                   tuple[str, ...] | set[str],
                   carpetas_disponibles: list[str],
                   raiz: Path = ORIGEN_LOCAL) -> tuple[dict[str, pd.DataFrame],
                                                      dict[str, str]]:
    """
    Función que lee los archivos csv que se encuentran dentro de la carpeta
    pasada como parametro y sus subcarpetas.
    Tambien se puede pasar una lista de carpetas.

    Es lectura para tratamiento de limpieza.
    Los csv se leen según los parametros encontrados en la deteccion de formato
    y codificacion, guardado en diccionario JSON importado previamente.
    """
    carpetas_comprobardas = comprobar_nombre_carpetas(carpeta,
                                                      carpetas_disponibles)
    rutas = sorted(
        r for c in carpetas_comprobardas for r in (raiz / c).rglob('*')
        if r.is_file() and r.suffix.lower() == '.csv'
        )
    dict_dfs, dict_fallos = leer_csv(params, rutas, raiz)
    resumen_valores_lectura(dict_dfs, dict_fallos)

    return dict_dfs, dict_fallos

## Limpieza inicial CSV

En esta primera limpieza se realizan las siguientes operaciones:
- Nombre columnas normalizados, en minusculas, sin acentos ni letras ñ, eliminación de los espacios extras en los campos y nombre unidos por barra baja (_).
- Normalización de los valores en minusculas, sin acentos, elimiación de los espacios extras, normalización sin acentos ni ñ.
- Detección valores nulos, sin asignarlos, para revisión posterior.

In [ ]:
# 1. MOSTRAR INFORMACION
def mostrar_inicial(nombre: str, df: pd.DataFrame) -> None:
    """
    Función que muestra la información principal del DF antes de ejecutar
    las transformaciones.
    Informacion: nombre tabla, .info con info de shape, dtype y
    count valores no nulos, una muestra (sample) de 3 registros y descripción
    inicial de todas las variables.
    """
    print(f'\n--------------- DF: {nombre} ---------------\n'
    "\n1. Información incial:\n"
    '\n1.1 Muestra:\n')
    display(df.sample(1))
    print("\n1.2 Información general:\n")
    df.info()
    print("\n1.3 Descripción inicial:\n")
    display(df.describe(include="all"))


def mostrar_normalizada(nombre: str, df: pd.DataFrame) -> None:
    """
    Recibe DF completo y normalizado.
    Muestra informacion post normalización: 1 registro y descripción de las
    las variables.
    """
    print(f"\n2. Resultados post normalizaciones del DF {nombre}:\n"
    f"2.1 Muestra:\n")
    display(df.sample(1))
    print("\n2.2 Descripción:\n")
    display(df.describe(include="all"))


def mostrar_posibles_nulos(nombre: str, df: pd.DataFrame | None) -> None:
    """
    Recibe DF con el resumen nulos.
    Muestra el resumen de los posibles valores nulos encontrados.
    Si None, es que no hay posibles nulos, imprime aviso.
    """
    print(f"\n3. Primera inspección posibles nulos de la tabla {nombre}:\n")
    if df is None:
      print(f"La tabla {nombre} no presenta posibles valores nulos.")
    else:
      print("\nResumen posibles nulos encontrado:\n")
      display(df)


# 2. NORMALIZAR Y CAMBIAR NOMBRES COLUMNAS
def normalizar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Función que normaliza los nombres de las columnas de cada df y los pasa a
    minusculas, uniendo separacion palabras con '_'.
    Guarda cambios realizados en diccionario e imprime lista de nombres duplicados,
    si hay.
    """
    nuevas = []
    for c in df.columns:
      c = str(c).replace("\ufeff", "").strip().lower()
      c = unicodedata.normalize("NFKD", c)
      c = "".join(ch for ch in c if not unicodedata.combining(ch))
      c = re.sub(r"[^0-9a-z]+", "_", c).strip("_")
      nuevas.append(c)

    df = df.set_axis(nuevas, axis=1)

    if df.columns.duplicated().any():
      raise ValueError(
          f"Columnas duplicadas: {df.columns[df.columns.duplicated()].tolist()}"
          )

    return df


# 3. NORMALIZAR VALORES DF
def normalizar_unicos(s: pd.Series) -> pd.Series:
    """
    Función para normalizar los valores unicos del DF.
    """
    s = (s.str.lower()
          .str.normalize("NFD") # Elimina acentos/dieresis y ñ
          .str.replace(r"[\u0300-\u036f]", "", regex=True)
          .str.strip()
          .str.replace(r"\s+", " ", regex=True))
    return s


def normalizar_texto(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza los valores del texto, primero los valores unicos y despues
    expande al resto.
    Se eliminan espacios en blanco y se pasa todo el texto a minusculas.
    """
    for c in df.select_dtypes("string").columns: # Filtra dtype string
      codigos, unicos = pd.factorize(df[c])
      unicos_norm = normalizar_unicos(pd.Series(unicos, dtype=df[c].dtype)).array
      df[c] = unicos_norm.take(codigos, allow_fill=True)

    return df


# 4. VER NULOS DETECCION DE NULOS IMPLICITOS / OCULTOS
def buscar_nulos(df: pd.DataFrame,
                 patrones: tuple[str] = PATRONES_NULOS) -> pd.DataFrame | None:
    """
    Función que busca patrones y valores nulos (patrones
    nulos) en la tabla.
    Si no encuentra, retorna None para aviso de que no hay.
    Si encuentra, los registra en una mascara y realiza resumen total
    de los valores encontrados
    por campo con posibles valores nulos y totales de cada valor.
    """
    cols = df.select_dtypes("string").columns # Para confirmar que cada columna es string
    mascara = df[cols].isin(patrones)
      # Mascara para comprobar presencia valores nulos y calcular resumen de ellos si hay

    if not mascara.any().any():
        return None

    largo = (
        df[cols].where(mascara)
        .melt(var_name="columna", value_name="valor")
        .dropna()
    )
    resumen = pd.crosstab(largo["valor"], largo["columna"])
    resumen = resumen.loc[[p for p in patrones if p in resumen.index]]
    resumen["total"] = resumen.sum(axis=1)
    resumen.loc["total"] = resumen.sum()
    resumen.index = [v if v == "total" else repr(v) for v in resumen.index]
    # entrecomilla los valores nulos para que se muestre el vacio
    del mascara, largo

    return resumen


# ULTITMA: FUNCION QUE ORQUESTA
def transformaciones_comunes(dfs: dict[str, pd.DataFrame]) -> tuple[
    dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    """
    Función que orquestra la muestra de información de los distintos DFs
    previos a su tratamiento y las primeras transformaciones y limpiezas.
    Los cambios realizados se guardaran sobre nuevo conjunto de DFs en nuevo
    diccionario.

    Por cada DF se mostrarará:
    - Info inicial (.info(shape, dtype, count valores no nulos), sample(3) y
      descripcion inicial de todas las variables)
    - Sample del DF normalizado nombres campos + texto, descripción.
    - De haberlos, resumen de los posibles errores, antes de
    convertirlos en nulos explicitos.

    Retorna diccionarios con claves con nombre de los archivos y valores:
    - DFs semilimpios.
    - Resumen nulos
    """

    dic_cambio_nombres = {}
    dfs_resumen_nulos = {} # Diccionario de DF con resumen de posibles nulos
    dfs_semilimpios = {} # Diccionario con los DFs transformados
    for nombre, df in dfs.items():
      mostrar_inicial(nombre, df)
      df = normalizar_columnas(df)
      df = normalizar_texto(df)
      mostrar_normalizada(nombre, df)
      dfs_resumen_nulos[nombre] = buscar_nulos(df)
      mostrar_posibles_nulos(nombre, dfs_resumen_nulos[nombre])
      dfs_semilimpios[nombre] = df

    return dfs_semilimpios, dfs_resumen_nulos

#